> 在线讲义：[第 60 章 · 感知的学习与评测：IoU、NMS、PR 曲线与 AP、匈牙利匹配、度量学习与地点识别](https://wikieden.github.io/math-and-eval-stats/lessons/60-perception-metrics.html) · 本书《数学与实验统计》

# 第 60 章 · 感知的学习与评测：IoU、NMS、PR 曲线与 AP、匈牙利匹配、度量学习与地点识别

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/60-perception-metrics.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 IoU | 交 / 并 | `iou(a, b)` | 0.7467 |
| 第二步 NMS | 贪心压制 | `nms()` | [0, 2] |
| 第三步 AP | 包络面积 | `ap()` | 0.6643 |
| 第四步 匹配 | 匈牙利 / 贪心 | `hungarian()`、`greedy()` | 7.5 vs 13.0 |
| 第五步 损失 | 三元组 / InfoNCE | `triplet`、`infonce` | 0.10；1.2153 → 0.0182 |

例子：两个框的 IoU；3 个重叠的检测框做 NMS；8 个检测、5 个真值的 PR 曲线和 AP；3 个预测对 3 个真值的匈牙利匹配
（和贪心匹配比较）；三元组损失；InfoNCE；NetVLAD 的维数。
里程碑 ME-M35 的“IoU、NMS、mAP、匈牙利匹配的完整实现和测试”留给自己做；这里每个只算一个小例子。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

**函数 `iou`**：框 (x1, y1, x2, y2)：交集面积 / 并集面积。

In [ ]:
def iou(a, b) -> float:
    """框 (x1, y1, x2, y2)：交集面积 / 并集面积。"""
    w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = w * h
    area = lambda r: (r[2] - r[0]) * (r[3] - r[1])  # noqa: E731
    return inter / (area(a) + area(b) - inter)

In [ ]:
BOXES = [((10, 10, 50, 50), 0.9), ((12, 14, 52, 54), 0.8), ((60, 10, 100, 50), 0.7), ((15, 12, 48, 49), 0.6)]

**函数 `nms`**：按分数从高到低，保留当前框，删掉和它 IoU > thr 的框。

In [ ]:
def nms(boxes=BOXES, thr: float = 0.5) -> list[int]:
    """按分数从高到低，保留当前框，删掉和它 IoU > thr 的框。"""
    order = sorted(range(len(boxes)), key=lambda i: -boxes[i][1])
    keep = []
    while order:
        i = order.pop(0)
        keep.append(i)
        order = [j for j in order if iou(boxes[i][0], boxes[j][0]) <= thr]
    return keep

In [ ]:
HITS = [1, 1, 0, 1, 0, 0, 1, 0]   # 按分数排好的 8 个检测是否命中（IoU ≥ 0.5 且没被更高分的检测占用）
N_GT = 5

**函数 `ap`**：精确率 P_k = 命中数/k，召回率 R_k = 命中数/真值数；AP = 精确率包络（右侧最大值）对召回率的面积。

In [ ]:
def ap(hits=HITS, n_gt: int = N_GT) -> tuple[list[float], list[float], float]:
    """精确率 P_k = 命中数/k，召回率 R_k = 命中数/真值数；AP = 精确率包络（右侧最大值）对召回率的面积。"""
    tp = np.cumsum(hits)
    prec = tp / np.arange(1, len(hits) + 1)
    rec = tp / n_gt
    env = np.maximum.accumulate(prec[::-1])[::-1]
    r_prev, area = 0.0, 0.0
    for r, p in zip(rec, env):
        area += (r - r_prev) * p
        r_prev = r
    return prec.round(4).tolist(), rec.round(4).tolist(), float(area)

In [ ]:
COST = np.array([[1.0, 2.0, 8.0], [1.5, 6.0, 9.0], [7.0, 3.0, 4.0]])

**函数 `hungarian`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def hungarian(cost=COST) -> tuple[list[int], float]:
    r, c = linear_sum_assignment(cost)
    return c.tolist(), float(cost[r, c].sum())

**函数 `greedy`**：每次挑全局最小的一对，划掉这一行一列。

In [ ]:
def greedy(cost=COST) -> tuple[list[int], float]:
    """每次挑全局最小的一对，划掉这一行一列。"""
    m = cost.copy()
    assign, total = [-1] * len(m), 0.0
    for _ in range(len(m)):
        i, j = np.unravel_index(np.argmin(m), m.shape)
        assign[i] = int(j)
        total += cost[i, j]
        m[i, :], m[:, j] = np.inf, np.inf
    return assign, total

**函数 `triplet`**：三元组损失 max(0, d(a, p) − d(a, n) + margin)。

In [ ]:
def triplet(da: float, dn: float, margin: float = 0.2) -> float:
    """三元组损失 max(0, d(a, p) − d(a, n) + margin)。"""
    return max(0.0, da - dn + margin)

**函数 `infonce`**：InfoNCE = −log( e^{s⁺/τ} / Σ_all e^{s/τ} )。

In [ ]:
def infonce(sim_pos: float, sim_negs: list[float], tau: float) -> float:
    """InfoNCE = −log( e^{s⁺/τ} / Σ_all e^{s/τ} )。"""
    z = np.array([sim_pos] + list(sim_negs)) / tau
    return float(-(z[0] - (z.max() + np.log(np.exp(z - z.max()).sum()))))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 IoU")
print(f"  (10, 10, 50, 50) 和 (12, 14, 52, 54)：IoU = {iou(BOXES[0][0], BOXES[1][0]):.4f}")
print(f"  (10, 10, 50, 50) 和 (60, 10, 100, 50)：IoU = {iou(BOXES[0][0], BOXES[2][0]):.4f}")

In [ ]:
print("\n第二步 NMS")
print(f"  4 个框（分数 0.9、0.8、0.7、0.6），阈值 0.5：保留 {nms()}")

In [ ]:
print("\n第三步 PR 曲线与 AP")
p, r, a = ap()
print(f"  命中序列 {HITS}、真值 {N_GT} 个：精确率 {p}")
print(f"  召回率 {r}；AP = {a:.4f}")

In [ ]:
print("\n第四步 匈牙利匹配 vs 贪心")
ha, hc = hungarian()
ga, gc = greedy()
print(f"  代价矩阵 {COST.tolist()}：匈牙利 {ha}，总代价 {hc}；贪心 {ga}，总代价 {gc}")

In [ ]:
print("\n第五步 度量学习：三元组损失与 InfoNCE")
for da, dn in ((0.3, 0.9), (0.5, 0.6), (0.8, 0.5)):
    print(f"  d(a, p) = {da}、d(a, n) = {dn}：三元组损失 {triplet(da, dn):.2f}")
for tau in (1.0, 0.1, 0.05):
    print(f"  正样本相似度 0.8、负样本 [0.6, 0.3, 0.1, 0.0]，τ = {tau}：InfoNCE = {infonce(0.8, [0.6, 0.3, 0.1, 0.0], tau):.4f}")
print(f"  NetVLAD：K = 64 个聚类中心、D = 512 维局部特征 → {64 * 512:,} 维全局描述子")

## 3. 代码里才看得到的

1. **命中序列是直接给的**：真实评测要先按 IoU ≥ 0.5 把检测和真值配对（每个真值只能被一个检测占用），这一步决定了 TP / FP。
2. **AP 用的是“所有点插值”**：PASCAL VOC 2007 用 11 点插值，COCO 用 101 点，三种算法的数值略有不同。
3. **每个指标只算了一个小例子**：IoU、NMS、mAP、匈牙利匹配的完整实现和测试留给自己做（里程碑 ME-M35）。

## 4. 自己改一改

1. 把 NMS 阈值改成 0.8，`nms()` 保留哪些框？
2. 把命中序列最后一个改成 1（第 5 个真值也被检出），AP 变成多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> IoU 交比并，过半才算中；
> NMS 按分数留高的、压重叠的，靠太近会误删；
> 排序算累计 P 和 R，AP 是包络下的面积，漏检截掉召回；
> 匈牙利一对一看全局，贪心会把别人逼到贵处；
> 度量学习拉近正、推远负，τ 小盯难负样本。